# Week 21, Day 2: Running scripts

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w021d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sys
print(sys.argv)
print(len(sys.argv))

In [ ]:
def main(argv):
    if len(argv) != 3:
        print("usage: convert.py TEMPERATURE UNIT")
        return 2
    value, unit = float(argv[1]), argv[2]
    if unit == "c":
        print(f"{value}°C is {value * 9 / 5 + 32:.1f}°F")
    else:
        print(f"{value}°F is {(value - 32) * 5 / 9:.1f}°C")
    return 0

# On your computer the last line would be: sys.exit(main(sys.argv))
main(["convert.py", "100", "c"])
main(["convert.py", "98.6", "f"])
main(["convert.py"])

In [ ]:
import argparse

def make_parser():
    parser = argparse.ArgumentParser(prog="convert", description="Convert temperatures.")
    parser.add_argument("value", type=float, help="the temperature to convert")
    parser.add_argument("--to", choices=["c", "f"], default="f", help="unit to convert to (default: f)")
    parser.add_argument("--places", type=int, default=1, help="decimal places")
    parser.add_argument("-v", "--verbose", action="store_true", help="explain the working")
    return parser

parser = make_parser()
args = parser.parse_args(["37", "--places", "2", "-v"])   # on your computer: parser.parse_args()
print(args)
print(args.value, args.to, args.places, args.verbose)
print(parser.format_help())

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="convert")
parser.add_argument("value", type=float)
try:
    parser.parse_args(["hot"])
except SystemExit as e:
    print("argparse stopped the program with exit code", e.code)

In [ ]:
import io

def average(lines):
    numbers = [float(line) for line in lines if line.strip()]
    return sum(numbers) / len(numbers)

fake_stdin = io.StringIO("71\n64\n\n90\n")
print(average(fake_stdin))
# On your computer: print(average(sys.stdin))

In [ ]:
import os

data_dir = os.environ.get("DATA_DIR", "./data")     # a default if it isn't set
print("reading from", data_dir)
print("HOME is set:", "HOME" in os.environ)

> **Key idea.** Run scripts with `python3 script.py`. Take information from the command line with `argparse` (or `sys.argv`), not `input()`. `print` goes to standard output; send errors to `sys.stderr`; end with `sys.exit(0)` for success or a non-zero code for failure. Read pipes from `sys.stdin`, and settings from `os.environ`. Put the work in functions that take their inputs as arguments, so they can be tested.

### Exercise: Arguments by hand

Write `main(argv)` for a script `repeat.py` used as `repeat.py WORD COUNT`: it returns the word repeated COUNT times, separated by spaces. If the number of arguments is wrong, or COUNT isn't a whole number, return the string `"usage: repeat.py WORD COUNT"`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert main([\"repeat.py\", \"hip\", \"3\"]) == \"hip hip hip\"\n    assert main([\"repeat.py\", \"go\", \"1\"]) == \"go\"\n    assert main([\"repeat.py\"]) == \"usage: repeat.py WORD COUNT\"\n    assert main([\"repeat.py\", \"x\", \"three\"]) == \"usage: repeat.py WORD COUNT\"\n    assert main([\"repeat.py\", \"a\", \"2\", \"extra\"]) == \"usage: repeat.py WORD COUNT\"")

### Exercise: An argparse interface

Write `make_parser()` returning an `ArgumentParser` for a script that resizes images: a positional `files` (one or more file names: `nargs="+"`), `--width` (a whole number, required: `required=True`), `--format` (one of `jpg`, `png`, `webp`, default `jpg`) and a flag `--dry-run` (`store_true`). Argparse turns `--dry-run` into `args.dry_run`.

In [ ]:
import argparse

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    p = make_parser()\n    a = p.parse_args([\"a.png\", \"b.png\", \"--width\", \"800\"])\n    assert a.files == [\"a.png\", \"b.png\"] and a.width == 800 and a.format == \"jpg\" and a.dry_run is False\n    b = p.parse_args([\"--width\", \"300\", \"c.jpg\", \"--format\", \"webp\", \"--dry-run\"])\n    assert b.files == [\"c.jpg\"] and b.format == \"webp\" and b.dry_run is True\n    for bad in ([\"a.png\"], [\"a.png\", \"--width\", \"big\"], [\"a.png\", \"--width\", \"5\", \"--format\", \"gif\"], [\"--width\", \"5\"]):\n        try:\n            p.parse_args(bad)\n            assert False, f\"{bad} should be rejected\"\n        except SystemExit as e:\n            assert e.code == 2")

### Exercise: A filter for pipes

Write `count_words(lines)` that takes any stream of lines (a list, `sys.stdin`, or an `io.StringIO`) and returns a tuple `(lines, words, characters)`, like the shell's `wc` command: characters counts everything, including the newlines. Then write `main(lines)` that returns the three numbers as one string, separated by spaces.

In [ ]:
import io

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    import io\n    assert count_words(io.StringIO(\"the cat sat\\non the mat\\n\")) == (2, 6, 23)\n    assert count_words([\"one\\n\", \"\\n\", \"two three\\n\"]) == (3, 3, 15)\n    assert count_words(io.StringIO(\"\")) == (0, 0, 0)\n    assert main(io.StringIO(\"a b\\n\")) == \"1 2 4\"")

### Exercise: Success or failure

Write `check_csv(lines)` that checks lines of a CSV file where every line must have the same number of comma-separated fields as the first (the header). It returns a tuple `(exit_code, messages)`: `(0, [])` if all is well; otherwise `1` and a list of messages like `"line 3: expected 3 fields, found 2"` (counting lines from 1). An empty file is an error with the message `"empty file"`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    good = [\"name,age,town\\n\", \"Ada,31,Leeds\\n\", \"Ben,25,Bath\\n\"]\n    assert check_csv(good) == (0, [])\n    bad = [\"name,age,town\\n\", \"Ada,31\\n\", \"Ben,25,Bath\\n\", \"Cy,40,York,UK\\n\"]\n    assert check_csv(bad) == (1, [\"line 2: expected 3 fields, found 2\", \"line 4: expected 3 fields, found 4\"])\n    assert check_csv([]) == (1, [\"empty file\"])")

### Exercise: Settings from the environment

Write `settings(environ)` that takes a dictionary of environment variables and returns a dictionary with `"data_dir"` (from `ATLAS_DATA`, default `"./data"`), `"debug"` (`True` if `ATLAS_DEBUG` is `"1"`, `"true"` or `"yes"`, ignoring case; otherwise `False`) and `"workers"` (from `ATLAS_WORKERS` as a whole number, default 4). Passing the dictionary in (rather than reading `os.environ` inside) makes it testable; the real script would call `settings(os.environ)`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert settings({}) == {\"data_dir\": \"./data\", \"debug\": False, \"workers\": 4}\n    assert settings({\"ATLAS_DATA\": \"/srv/d\", \"ATLAS_DEBUG\": \"Yes\", \"ATLAS_WORKERS\": \"8\"}) == {\"data_dir\": \"/srv/d\", \"debug\": True, \"workers\": 8}\n    assert settings({\"ATLAS_DEBUG\": \"0\"})[\"debug\"] is False")

### Bug hunt: Strings, not numbers

`add.py` should add up the numbers given on the command line, but it joins them into one long string: for `add.py 2 3 4` it gives `234`. Fix `main` so it returns `9` for that.

In [ ]:
def main(argv):
    total = ""
    for word in argv[1:]:
        total += word
    return total

print(main(["add.py", "2", "3", "4"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"9\"\n    expect_output(\"9\")\n    assert main([\"add.py\"]) == 0 and main([\"add.py\", \"1.5\", \"2\"]) == 3.5")

### Bug hunt: Errors in the output

This script prints its error message to standard output, so when someone runs `python3 grades.py > report.txt`, the error ends up hidden inside the report. Send the error line to standard error instead (the report line must stay on standard output), so that only `Ada: 71` goes to standard output. (On this site, both streams appear in the output panel; the checks look at them separately.)

In [ ]:
import sys

marks = {"Ada": "71", "Ben": "absent"}
for name, mark in marks.items():
    if mark.isdigit():
        print(f"{name}: {mark}")
    else:
        print(f"warning: no mark for {name}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only the report on stdout\"\n    import io, contextlib, sys\n    out, err = io.StringIO(), io.StringIO()\n    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):\n        exec(compile(source, \"<program>\", \"exec\"), {\"__name__\": \"__main__\"})\n    assert out.getvalue().strip() == \"Ada: 71\", \"standard output should hold only the report\"\n    assert \"warning: no mark for Ben\" in err.getvalue(), \"the warning should go to standard error\"")

### Puzzle: The shape of a script

Arrange the lines into a script that greets the name given on the command line, with a proper exit code. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # sys.argv for: python3 tool.py --fast data.csv  (a list)
b = ___   # the exit code that means success
c = ___   # the type of every item in sys.argv (as a word: "str", "int" ...)
d = ___   # the shell symbol that sends one program's output into another's input

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [\"tool.py\", \"--fast\", \"data.csv\"] and b == 0 and c == \"str\" and d == \"|\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a complete command-line tool

Build `todo`, a to-do list tool, as a function `main(argv, store)` that works like a real script. `argv` is the list of arguments **after** the program name, and `store` is a list of task strings (the "file" the tool keeps its tasks in; the function changes it in place). Use `argparse` with **subcommands** (`parser.add_subparsers(dest="command", required=True)`):

- `add TEXT...` adds one task (the words joined by spaces); output `Added 3: buy eggs` (its number, counting from 1).
- `list` outputs each task as `1. buy eggs`, or `Nothing to do.` if empty.
- `done N` removes task N; output `Done: buy eggs`. If N doesn't exist, output `No task N` with exit code 1.
- `clear --yes` removes everything (without `--yes`, output `Add --yes to really clear` with exit code 1).

`main` returns `(exit_code, output_lines)`. Bad usage (like an unknown command) must give exit code 2, which argparse produces by raising `SystemExit(2)`: catch it and return `(2, [])`. (argparse also prints a usage message; that's fine.)

In [ ]:
import argparse

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_flow():\n    \"a session\"\n    store = []\n    assert main([\"list\"], store) == (0, [\"Nothing to do.\"])\n    assert main([\"add\", \"buy\", \"eggs\"], store) == (0, [\"Added 1: buy eggs\"])\n    assert main([\"add\", \"call\", \"Ada\"], store) == (0, [\"Added 2: call Ada\"])\n    assert main([\"list\"], store) == (0, [\"1. buy eggs\", \"2. call Ada\"])\n    assert main([\"done\", \"1\"], store) == (0, [\"Done: buy eggs\"]) and store == [\"call Ada\"]\n    assert main([\"done\", \"5\"], store) == (1, [\"No task 5\"])\n    assert main([\"clear\"], store) == (1, [\"Add --yes to really clear\"]) and store == [\"call Ada\"]\n    assert main([\"clear\", \"--yes\"], store) == (0, [\"Cleared 1 task(s)\"]) and store == []\ndef test_usage():\n    \"bad usage\"\n    assert main([\"fly\"], []) == (2, [])\n    assert main([], []) == (2, [])\n    assert main([\"done\", \"two\"], []) == (2, [])")

## Warm-up

`sys.argv` is the list of words that were typed to start the program. Here, in the browser, there's only one: the program's own name. Run from a terminal as `python3 greet.py Ada --loud`, it would be `["greet.py", "Ada", "--loud"]`.

## Running a script

On your own computer, in the folder you made yesterday:

```
cd ~/atlas/week21
echo "print('Hello from a script')" > hello.py
python3 hello.py
```

(On Windows: `python hello.py` or `py hello.py`.) Python runs the file from top to bottom, exactly as the site does, and prints to the terminal. Write longer scripts in a code editor (Visual Studio Code is free and widely used; Week 22 sets one up properly), save them, and run them from the terminal.

A few useful ways of starting Python:

| Command | Does |
|---|---|
| `python3 script.py` | runs a script |
| `python3` | starts the interactive prompt `>>>` (leave with `exit()`) |
| `python3 -c "print(2 ** 100)"` | runs one line |
| `python3 -m module` | runs a module as a script, e.g. `python3 -m http.server` serves the current folder as a website, `python3 -m json.tool data.json` pretty-prints JSON |
| `python3 -i script.py` | runs a script, then stays at the `>>>` prompt with its variables, for exploring |

## sys.argv

Scripts that ask questions with `input()` can't be automated: someone has to sit and type. Instead, scripts usually take their information from the **command line**:

```
python3 convert.py 100 c
```

`sys.argv` holds the words: `["convert.py", "100", "c"]`. Every item is a **string**, so numbers must be converted.

Writing `main(argv)` to take the list as an argument, rather than reading `sys.argv` inside, means you can call it with any list you like, which is how the examples here (and tests) can try it out without a terminal.

## argparse

Handling arguments by hand gets messy fast: optional flags, defaults, checking types, help messages. The standard `argparse` module does all of it:

- **Positional** arguments (`"value"`) are required and given in order.
- **Optional** arguments start with `--` (a long name) or `-` (a short one), can have a `default`, and can be restricted with `choices`.
- `type=float` converts the text, and gives a clear error if it can't.
- `action="store_true"` makes a **flag**: `True` if it's given, `False` if not.
- `--help` (or `-h`) prints the help text automatically.

`parse_args()` with no list reads `sys.argv`. Given bad input, argparse prints a usage message and stops the script with exit code 2. In the browser that shows up as `SystemExit`:

## Input, output, errors and exit codes

Every program has three standard streams:

- **standard input** (stdin): what's typed, or piped in;
- **standard output** (stdout): `print` writes here;
- **standard error** (stderr): for error messages and warnings, kept separate so they don't get mixed into the real output. Write to it with `print(..., file=sys.stderr)`.

**Exit codes.** When a program ends, it hands the shell a number: **0 means success**, anything else means something went wrong (by convention, 1 for a general error and 2 for bad usage). `sys.exit(code)` ends the script with that code. Other programs and scripts check it to decide what to do next; in a shell, `echo $?` shows the last one.

**Pipes.** The shell's `|` connects one program's standard output to the next program's standard input:

```
cat marks.txt | python3 average.py
python3 make_report.py > report.txt
python3 check.py 2> errors.txt
```

To read standard input in Python, loop over `sys.stdin`, which gives one line at a time:

```
import sys
total = count = 0
for line in sys.stdin:
    total += float(line)
    count += 1
print(total / count)
```

To make code like this testable, write the work in a function that takes **any** stream of lines, and pass it `sys.stdin` on the real command line, or an `io.StringIO` (a string that behaves like a file) in a test:

## Environment variables

The shell keeps a set of named settings called **environment variables**, which every program it starts can read: `HOME` (your home folder), `PATH` (the folders searched for commands), `USER` and many more. Scripts often read their own settings this way, especially secrets like API keys that must never be written into the code:

```
export DATA_DIR=/home/you/data      (macOS and Linux)
$env:DATA_DIR = "C:\data"           (PowerShell)
```

`os.environ` is a dictionary-like object. Use `.get(name, default)`, since a missing variable is common.

**`PATH`** explains a puzzle you may have met yesterday: typing `python3` works because the folder containing the `python3` program is listed in `PATH`. "command not found" usually means it isn't.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, save today's `todo` tool as `todo.py`, with its tasks kept in a real file (`tasks.txt`, one per line), and end it with `sys.exit(code)` after printing the output lines. Use it from your terminal for a day. Tomorrow: Git, which records every version of your work so you can never lose it.